In [1]:
import gc
from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs

# Configure Polars to print everything
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns


polars.config.Config

In [2]:
# I/O Constants
INPUT_BASE_PATH = Path("./data/raw")
OUTPUT_BASE_PATH = Path("./data")

# Data constants
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
N_EVENTS = 2
ELECTION_COLS = ["snap_CA", "snap_TX", "snap_WI"]
MAX_TRAIN_TIMESTAMP = 1941

# Feature engineering constants
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]
PRICE_LAGS = [1, 2, 3, 4, 7]
ROLLING_PRICE_WINDOWS = [2, 3, 4, 7, 14]

MAX_DAYS_UNTIL_EVENT = 7
MAX_DAYS_SINCE_EVENT = 7
MAX_DAYS_UNTIL_ELECTION = 7
MAX_DAYS_SINCE_ELECTION = 7

### Load data

In [3]:
# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in ["CA", "TX", "WI"]}
}
raw_calendar_df = pl.read_csv(
    f"{INPUT_BASE_PATH}/calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell prices
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32,
}
raw_sell_price_df = pl.read_csv(
    f"{INPUT_BASE_PATH}/sell_prices.csv",
    schema_overrides=sell_price_schema_overrides
)

# Sales data
sales_schema_overrides = {
    f"d_{t}": pl.Int16
    for t in range(1, MAX_TRAIN_TIMESTAMP + 1)
}
raw_sales_df = pl.read_csv(
    INPUT_BASE_PATH / "sales_train_evaluation.csv",
    schema_overrides=sales_schema_overrides,
)

# Sample submission
sample_submission_df = pl.read_csv(
    INPUT_BASE_PATH / "sample_submission.csv",
)

In [4]:
# Merge all raw dataframes into feature df.

feature_df = (
    raw_sales_df
    # Cast keys to categorical on the small wide frame, before unpivoting to ~59M rows
    .with_columns(pl.col(ID_COLS).cast(pl.Categorical))
    .unpivot(
        index=ID_COLS,
        variable_name="d",
        value_name="sales",
    )
    .with_columns(
        pl.col("sales").cast(pl.Int16),
        d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
    )
    # Drop string d column, no longer needed.
    # d_index is the key from here on
    .drop("d")
    .join(
        # Initially only joining on a subset of columns
        other=raw_calendar_df.select(
            "wm_yr_wk",
            d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16),
        ),
        on="d_index",
        how="left",
    )
    .join(
        other=raw_sell_price_df.with_columns(pl.col("store_id", "item_id").cast(pl.Categorical)),
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left",
    )
)

In [5]:
# Raw sales is no longer needed, free it from memory

del raw_sales_df
gc.collect()


13

In [6]:
# Remove rows where price is null
feature_df = feature_df.filter(pl.col("sell_price").is_not_null())

# Check no null prices
assert not feature_df["sell_price"].is_null().any()

In [7]:
# Check no duplicates

assert feature_df.select(pl.col("id"), pl.col("d_index")).is_duplicated().sum() == 0

In [8]:
# Check that timeseries are continuous

id_series_lengths = (
    feature_df
    .group_by("id")
    .agg(
        d_index_min=pl.col("d_index").min(),
        d_index_max=pl.col("d_index").max(),
    )
    .with_columns(min_max_diff=pl.col("d_index_max") - pl.col("d_index_min"))
    .join(
        other=feature_df.group_by("id").len(name="series_length"),
        on="id",
        how="inner",
        validate="1:1"
    )
)

assert id_series_lengths["min_max_diff"].eq(id_series_lengths["series_length"] - 1).all()

### Lagged sales features

In [9]:
# Add columns directly on main features_df, to avoid expensive joins later on

feature_df = feature_df.with_columns(
    [
        pl.col("sales")
        .shift(lag)
        .over("id", order_by="d_index")
        .alias(f"sales_lagged_{lag}")
        for lag in SALES_LAGS
    ]
)


(
    feature_df
    .select(["id", "d_index", "sales", *[f"sales_lagged_{lag}" for lag in SALES_LAGS]])
    .sort(by=["id", "d_index"])
    .head(max(SALES_LAGS) + 2)
)


id,d_index,sales,sales_lagged_1,sales_lagged_2,sales_lagged_3,sales_lagged_4,sales_lagged_7,sales_lagged_14
cat,i16,i16,i16,i16,i16,i16,i16,i16
"""FOODS_1_001_CA_1_evaluation""",1,3,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,0,3,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,0,0,3,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,1,0,0,3,null,null,null
"""FOODS_1_001_CA_1_evaluation""",5,4,1,0,0,3,null,null
"""FOODS_1_001_CA_1_evaluation""",6,2,4,1,0,0,null,null
"""FOODS_1_001_CA_1_evaluation""",7,0,2,4,1,0,null,null
"""FOODS_1_001_CA_1_evaluation""",8,2,0,2,4,1,3,null
"""FOODS_1_001_CA_1_evaluation""",9,0,2,0,2,4,0,null


### Rolling sales features

In [10]:
# Add columns directly on main features_df, to avoid expensive joins later on

feature_df = feature_df.with_columns(
    # Rolling means
    [
        pl.col("sales")
        .shift(1)
        .rolling_mean(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_mean_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
    # Rolling std. dev.
    + [
        pl.col("sales")
        .shift(1)
        .rolling_std(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_std_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
    # Rolling min / max
    + [
        pl.col("sales")
        .shift(1)
        .rolling_min(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_min_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
    + [
        pl.col("sales")
        .shift(1)
        .rolling_max(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sales_rolling_max_{window}")
        .cast(pl.Float32)
        for window in ROLLING_SALES_WINDOWS
    ]
)


(
    feature_df
    .select(
        ["id", "d_index", "sales"]
        + [f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS]
        + [f"sales_rolling_std_{w}" for w in ROLLING_SALES_WINDOWS]
        + [f"sales_rolling_min_{w}" for w in ROLLING_SALES_WINDOWS]
        + [f"sales_rolling_max_{w}" for w in ROLLING_SALES_WINDOWS]
    )
    .sort(by=["id", "d_index"])
    .head(max(SALES_LAGS) + 2)
)


id,d_index,sales,sales_rolling_mean_2,sales_rolling_mean_3,sales_rolling_mean_4,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_std_2,sales_rolling_std_3,sales_rolling_std_4,sales_rolling_std_7,sales_rolling_std_14,sales_rolling_min_2,sales_rolling_min_3,sales_rolling_min_4,sales_rolling_min_7,sales_rolling_min_14,sales_rolling_max_2,sales_rolling_max_3,sales_rolling_max_4,sales_rolling_max_7,sales_rolling_max_14
cat,i16,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""FOODS_1_001_CA_1_evaluation""",1,3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,0,1.5,null,null,null,null,2.12132,null,null,null,null,0.0,null,null,null,null,3.0,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,1,0.0,1.0,null,null,null,0.0,1.732051,null,null,null,0.0,0.0,null,null,null,0.0,3.0,null,null,null
"""FOODS_1_001_CA_1_evaluation""",5,4,0.5,0.333333,1.0,null,null,0.707107,0.57735,1.414214,null,null,0.0,0.0,0.0,null,null,1.0,1.0,3.0,null,null
"""FOODS_1_001_CA_1_evaluation""",6,2,2.5,1.666667,1.25,null,null,2.12132,2.081666,1.892969,null,null,1.0,0.0,0.0,null,null,4.0,4.0,4.0,null,null
"""FOODS_1_001_CA_1_evaluation""",7,0,3.0,2.333333,1.75,null,null,1.414214,1.527525,1.707825,null,null,2.0,1.0,0.0,null,null,4.0,4.0,4.0,null,null
"""FOODS_1_001_CA_1_evaluation""",8,2,1.0,2.0,1.75,1.428571,null,1.414214,2.0,1.707825,1.618347,null,0.0,0.0,0.0,0.0,null,2.0,4.0,4.0,4.0,null
"""FOODS_1_001_CA_1_evaluation""",9,0,1.0,1.333333,2.0,1.285714,null,1.414214,1.154701,1.632993,1.496027,null,0.0,0.0,0.0,0.0,null,2.0,2.0,4.0,4.0,null


### Lagged price features

In [11]:
# Add columns directly on main features_df, to avoid expensive joins later on

feature_df = feature_df.with_columns(
    [
        pl.col("sell_price")
        .shift(lag)
        .over("id", order_by="d_index")
        .alias(f"sell_price_lagged_{lag}")
        for lag in PRICE_LAGS
    ]
)


(
    feature_df
    .select(["id", "d_index", "sell_price", *[f"sell_price_lagged_{lag}" for lag in PRICE_LAGS]])
    .sort(by=["id", "d_index"])
    .head(max(PRICE_LAGS) + 2)
)

id,d_index,sell_price,sell_price_lagged_1,sell_price_lagged_2,sell_price_lagged_3,sell_price_lagged_4,sell_price_lagged_7
cat,i16,f32,f32,f32,f32,f32,f32
"""FOODS_1_001_CA_1_evaluation""",1,2.0,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,2.0,2.0,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,2.0,2.0,2.0,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,2.0,2.0,2.0,2.0,null,null
"""FOODS_1_001_CA_1_evaluation""",5,2.0,2.0,2.0,2.0,2.0,null
"""FOODS_1_001_CA_1_evaluation""",6,2.0,2.0,2.0,2.0,2.0,null
"""FOODS_1_001_CA_1_evaluation""",7,2.0,2.0,2.0,2.0,2.0,null
"""FOODS_1_001_CA_1_evaluation""",8,2.0,2.0,2.0,2.0,2.0,2.0
"""FOODS_1_001_CA_1_evaluation""",9,2.0,2.0,2.0,2.0,2.0,2.0


### Rolling price features

In [12]:
# Add columns directly on main features_df, to avoid expensive joins later on
# NOTE: Keeping a shift(1) in these features. Not technically required since
# prices are known in advance.

feature_df = feature_df.with_columns(
    [
        pl.col("sell_price")
        .shift(1)
        .rolling_mean(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sell_price_rolling_mean_{window}")
        .cast(pl.Float32)
        for window in ROLLING_PRICE_WINDOWS
    ]
    + [
        pl.col("sell_price")
        .shift(1)
        .rolling_std(window_size=window, min_samples=window)
        .over("id", order_by="d_index")
        .alias(f"sell_price_rolling_std_{window}")
        .cast(pl.Float32)
        for window in ROLLING_PRICE_WINDOWS
    ]
)


(
    feature_df
    .select(
        [
            "id", "d_index", "sell_price",
            *[f"sell_price_rolling_mean_{w}" for w in ROLLING_PRICE_WINDOWS],
            *[f"sell_price_rolling_std_{w}" for w in ROLLING_PRICE_WINDOWS],
        ]
    )
    .sort(by=["id", "d_index"])
    .head(max(ROLLING_PRICE_WINDOWS) + 2)
)

id,d_index,sell_price,sell_price_rolling_mean_2,sell_price_rolling_mean_3,sell_price_rolling_mean_4,sell_price_rolling_mean_7,sell_price_rolling_mean_14,sell_price_rolling_std_2,sell_price_rolling_std_3,sell_price_rolling_std_4,sell_price_rolling_std_7,sell_price_rolling_std_14
cat,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""FOODS_1_001_CA_1_evaluation""",1,2.0,null,null,null,null,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",2,2.0,null,null,null,null,null,null,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",3,2.0,2.0,null,null,null,null,0.0,null,null,null,null
"""FOODS_1_001_CA_1_evaluation""",4,2.0,2.0,2.0,null,null,null,0.0,0.0,null,null,null
"""FOODS_1_001_CA_1_evaluation""",5,2.0,2.0,2.0,2.0,null,null,0.0,0.0,0.0,null,null
"""FOODS_1_001_CA_1_evaluation""",6,2.0,2.0,2.0,2.0,null,null,0.0,0.0,0.0,null,null
"""FOODS_1_001_CA_1_evaluation""",7,2.0,2.0,2.0,2.0,null,null,0.0,0.0,0.0,null,null
"""FOODS_1_001_CA_1_evaluation""",8,2.0,2.0,2.0,2.0,2.0,null,0.0,0.0,0.0,0.0,null
"""FOODS_1_001_CA_1_evaluation""",9,2.0,2.0,2.0,2.0,2.0,null,0.0,0.0,0.0,0.0,null


### Event features

In [13]:
event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in range(1, N_EVENTS + 1)
    )
    # Days until next / since last event
    .with_columns(
        # Days since last event
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        # Last event type
        *[
            (pl.col(f"event_type_{i}").forward_fill()
            .alias(f"last_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
        # Days until next event
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        # Next event type
        *[
            (pl.col(f"event_type_{i}").backward_fill()
            .alias(f"next_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since last event and event type to MAX_DAYS_SINCE_EVENT.
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT).
            then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        # Clip days until next event and event type to MAX_DAYS_UNTIL_EVENT.
        *[              
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ]
    )
    .drop([f"event_index_{i}" for i in range(1, N_EVENTS + 1)])
)

event_features_df.head()

wm_yr_wk,d,date,event_name_1,event_type_1,event_name_2,event_type_2,d_index,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2
i16,str,date,str,str,str,str,i16,i16,i16,str,str,i16,i16,str,str
11101,"""d_1""",2011-01-29,null,null,null,null,1,null,null,null,null,null,null,null,null
11101,"""d_2""",2011-01-30,null,null,null,null,2,null,null,null,null,7,null,"""Sporting""",null
11101,"""d_3""",2011-01-31,null,null,null,null,3,null,null,null,null,6,null,"""Sporting""",null
11101,"""d_4""",2011-02-01,null,null,null,null,4,null,null,null,null,5,null,"""Sporting""",null
11101,"""d_5""",2011-02-02,null,null,null,null,5,null,null,null,null,4,null,"""Sporting""",null


### Calendar features

In [14]:
calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

calendar_features_df.head()

wm_yr_wk,d,date,wday,month,year,d_index,sin_wday,cos_wday,sin_month,cos_month,is_weekend
i16,str,date,i16,i16,i16,i16,f32,f32,f32,f32,f32
11101,"""d_1""",2011-01-29,1,1,2011,1,0.781832,0.62349,0.5,0.866025,1.0
11101,"""d_2""",2011-01-30,2,1,2011,2,0.974928,-0.222521,0.5,0.866025,1.0
11101,"""d_3""",2011-01-31,3,1,2011,3,0.433884,-0.900969,0.5,0.866025,0.0
11101,"""d_4""",2011-02-01,4,2,2011,4,-0.433884,-0.900969,0.866025,0.5,0.0
11101,"""d_5""",2011-02-02,5,2,2011,5,-0.974928,-0.222521,0.866025,0.5,0.0


### Election features

In [15]:
election_features_df = (
    raw_calendar_df
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(election).eq(1))
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"{election}_index")
        for election in ELECTION_COLS
    )
    .select(["wm_yr_wk", "date", "d", "d_index"] + ELECTION_COLS + [f"{e}_index" for e in ELECTION_COLS])
    .with_columns(
        # Days since last election
        *[
            (pl.col("d_index") - pl.col(f"{election}_index").forward_fill())
            .alias(f"days_since_{election}")
            for election in ELECTION_COLS
        ],
        # Days until next election
        *[
            (pl.col(f"{election}_index").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_{election}")
            for election in ELECTION_COLS
        ]
    )
    .with_columns(
        # Clip days since/until election to MAX_DAYS_SINCE_ELECTION / MAX_DAYS_UNTIL_ELECTION
        *[
            pl.when(pl.col(f"days_since_{election}") <= MAX_DAYS_SINCE_ELECTION)
            .then(pl.col(f"days_since_{election}"))
            .otherwise(None)
            for election in ELECTION_COLS
        ],
        *[
            pl.when(pl.col(f"days_until_{election}") <= MAX_DAYS_UNTIL_ELECTION)
            .then(pl.col(f"days_until_{election}"))
            .otherwise(None)
            for election in ELECTION_COLS
        ]
    )
    .drop([f"{election}_index" for election in ELECTION_COLS])
)

election_features_df.head()

wm_yr_wk,date,d,d_index,snap_CA,snap_TX,snap_WI,days_since_snap_CA,days_since_snap_TX,days_since_snap_WI,days_until_snap_CA,days_until_snap_TX,days_until_snap_WI
i16,date,str,i16,i16,i16,i16,i16,i16,i16,i16,i16,i16
11101,2011-01-29,"""d_1""",1,0,0,0,null,null,null,3,3,4
11101,2011-01-30,"""d_2""",2,0,0,0,null,null,null,2,2,3
11101,2011-01-31,"""d_3""",3,0,0,0,null,null,null,1,1,2
11101,2011-02-01,"""d_4""",4,1,1,0,0,0,null,0,0,1
11101,2011-02-02,"""d_5""",5,1,0,1,0,1,0,0,1,0


## MERGE ALL TRAIN FEATURES

In [16]:
# Merge small calendar and event features df first, drop redunant columns and
# cast to categorical before merging onto large features df.
events_calendar_elections_features_df = (
    calendar_features_df.drop("d", "date")
    .join(
        other=event_features_df.drop("d", "date"),
        on=["wm_yr_wk", "d_index"],
        how="inner",
        validate="1:1",
    )
    .join(
        other=election_features_df.drop("d", "date"),
        on=["wm_yr_wk", "d_index"],
        how="inner",
        validate="1:1",
    )
    .with_columns(cs.string().cast(pl.Categorical))
)

full_train_feature_df = feature_df.join(
    other=events_calendar_elections_features_df,
    on=["wm_yr_wk", "d_index"],
    how="inner",
    validate="m:1",
)

full_train_feature_df.head()

id,item_id,dept_id,cat_id,store_id,state_id,sales,d_index,wm_yr_wk,sell_price,sales_lagged_1,sales_lagged_2,sales_lagged_3,sales_lagged_4,sales_lagged_7,sales_lagged_14,sales_rolling_mean_2,sales_rolling_mean_3,sales_rolling_mean_4,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_std_2,sales_rolling_std_3,sales_rolling_std_4,sales_rolling_std_7,sales_rolling_std_14,sales_rolling_min_2,sales_rolling_min_3,sales_rolling_min_4,sales_rolling_min_7,sales_rolling_min_14,sales_rolling_max_2,sales_rolling_max_3,sales_rolling_max_4,sales_rolling_max_7,sales_rolling_max_14,sell_price_lagged_1,sell_price_lagged_2,sell_price_lagged_3,sell_price_lagged_4,sell_price_lagged_7,sell_price_rolling_mean_2,sell_price_rolling_mean_3,sell_price_rolling_mean_4,sell_price_rolling_mean_7,sell_price_rolling_mean_14,sell_price_rolling_std_2,sell_price_rolling_std_3,sell_price_rolling_std_4,sell_price_rolling_std_7,sell_price_rolling_std_14,wday,month,year,sin_wday,cos_wday,sin_month,cos_month,is_weekend,event_name_1,event_type_1,event_name_2,event_type_2,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2,snap_CA,snap_TX,snap_WI,days_since_snap_CA,days_since_snap_TX,days_since_snap_WI,days_until_snap_CA,days_until_snap_TX,days_until_snap_WI
cat,cat,cat,cat,cat,cat,i16,i16,i16,f32,i16,i16,i16,i16,i16,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i16,i16,i16,f32,f32,f32,f32,f32,cat,cat,cat,cat,i16,i16,cat,cat,i16,i16,cat,cat,i16,i16,i16,i16,i16,i16,i16,i16,i16
"""HOBBIES_1_008_CA_1_evaluation""","""HOBBIES_1_008""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",12,1,11101,0.46,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,1,2011,0.781832,0.62349,0.5,0.866025,1.0,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,null,null,3,3,4
"""HOBBIES_1_009_CA_1_evaluation""","""HOBBIES_1_009""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",2,1,11101,1.56,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,1,2011,0.781832,0.62349,0.5,0.866025,1.0,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,null,null,3,3,4
"""HOBBIES_1_010_CA_1_evaluation""","""HOBBIES_1_010""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",0,1,11101,3.17,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,1,2011,0.781832,0.62349,0.5,0.866025,1.0,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,null,null,3,3,4
"""HOBBIES_1_012_CA_1_evaluation""","""HOBBIES_1_012""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",0,1,11101,5.98,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,1,2011,0.781832,0.62349,0.5,0.866025,1.0,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,null,null,3,3,4
"""HOBBIES_1_015_CA_1_evaluation""","""HOBBIES_1_015""","""HOBBIES_1""","""HOBBIES""","""CA_1""","""CA""",4,1,11101,0.7,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,1,2011,0.781832,0.62349,0.5,0.866025,1.0,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,null,null,3,3,4


In [17]:
# Write to output

full_train_feature_df.write_parquet(OUTPUT_BASE_PATH / "m5-acc-features-df.pq")

## TRAIN MODEL

## MERGE ALL SUBMISSION FEATURES

In [18]:
FORECAST_HORIZON = 28
MAX_TRAIN_INDEX = 1941
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)


FORECAST_IDS = (
    sample_submission_df
    .filter(sample_submission_df["id"].str.ends_with("_evaluation"))
    ["id"]
    .unique()
    .to_list()
)

In [19]:
# For submissions need to construct features iteratively.

# 1. Take final slice from train df needed to construct 
# features for first submision timestamp

max_lookback = max(SALES_LAGS + ROLLING_SALES_WINDOWS + PRICE_LAGS + ROLLING_PRICE_WINDOWS)
sales_history_df = (
    full_train_feature_df.select(["id", "d_index", "sales"])
    .sort(by=["id", "d_index"])
    .group_by("id")
    .tail(max_lookback)
    .sort(by=["id", "d_index"])
)

# Price history, extended with each forecast timestep's (known) prices inside the loop.
price_history_df = (
    full_train_feature_df.select(["id", "d_index", "sell_price"])
    .sort(by=["id", "d_index"])
    .group_by("id")
    .tail(max_lookback)
    .sort(by=["id", "d_index"])
)

sales_history_df.head(max_lookback + 2)

id,d_index,sales
cat,i16,i16
"""FOODS_1_001_CA_1_evaluation""",1928,0
"""FOODS_1_001_CA_1_evaluation""",1929,2
"""FOODS_1_001_CA_1_evaluation""",1930,0
"""FOODS_1_001_CA_1_evaluation""",1931,0
"""FOODS_1_001_CA_1_evaluation""",1932,2
"""FOODS_1_001_CA_1_evaluation""",1933,3
"""FOODS_1_001_CA_1_evaluation""",1934,1
"""FOODS_1_001_CA_1_evaluation""",1935,0
"""FOODS_1_001_CA_1_evaluation""",1936,0


In [20]:
# 2. Iterate over forecast timestamps, save forecasts, and
# construct features for next timestamp.

all_forecast_dfs: list[pl.DataFrame] = []

for f_index in range(1, FORECAST_HORIZON + 1):
    d_index = MAX_TRAIN_INDEX + f_index
    
    # Construct test_df with identifier columns
    test_df = (
        pl.DataFrame({"id": FORECAST_IDS})
        .with_columns(parsed_id=pl.col("id").str.extract_groups(_SERIES_ID_RE))
        .with_columns(
            item_id=pl.format(
                "{}_{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
                pl.col("parsed_id").struct.field("item"),
            ),
            dept_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
            ),
            cat_id=pl.format(
                "{}",
                pl.col("parsed_id").struct.field("cat"),
            ),
            store_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("state"),
                pl.col("parsed_id").struct.field("store"),
            ),
            state_id=pl.format(
                "{}",
                pl.col("parsed_id").struct.field("state"),
            )
        )
        .drop("parsed_id")
        .with_columns(
            id=pl.col("id").cast(pl.Categorical),
            d_index=pl.lit(d_index).cast(pl.Int16),
            sales=pl.lit(None).cast(pl.Int16)
        )
        # Prices for current d_index are known in advance, so look up the sell 
        # price for the forecast timestamp.
        .join(
            other=calendar_features_df.select(["d_index", "wm_yr_wk"]),
            on="d_index",
            how="left",
            validate="m:1",
        )
        .join(
            other=raw_sell_price_df,
            on=["item_id", "store_id", "wm_yr_wk"],
            how="left",
            validate="m:1",
        )
        .drop("wm_yr_wk")
    )

    # Construct lagged sales features
    previous_sales_df = pl.concat(
        [
            sales_history_df,
            *[
                df
                .select(["id", "d_index", "sales"])
                .with_columns(id=pl.col("id").cast(pl.Categorical))
                for df in all_forecast_dfs
            ],
            test_df.select(["id", "d_index", "sales"])
        ]
    )

    # Construct lagged and rolling sales features
    sales_features_test_df = (
        previous_sales_df
        .sort(by=["id", "d_index"])
        .with_columns(
            # Lagged sales
            [
                pl.col("sales")
                .shift(lag)
                .over("id", order_by="d_index")
                .alias(f"sales_lagged_{lag}")
                for lag in SALES_LAGS
            ]
            # Rolling mean sales
            + [
                pl.col("sales")
                .shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
            # Rolling std. dev. sales
            + [
                pl.col("sales")
                .shift(1)
                .rolling_std(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_std_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
            # Rolling min / max sales
            + [
                pl.col("sales")
                .shift(1)
                .rolling_min(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_min_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
            + [
                pl.col("sales")
                .shift(1)
                .rolling_max(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_max_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
        )
    )

    # Construct lagged and rolling price features
    previous_price_df = (
        pl.concat(
            [
                price_history_df,
                test_df.select(["id", "d_index", "sell_price"])
            ]
        )
        .sort(by=["id", "d_index"])
    )
    price_history_df = previous_price_df

    price_features_test_df = (
        previous_price_df
        .with_columns(
            [
                pl.col("sell_price")
                .shift(lag)
                .over("id", order_by="d_index")
                .alias(f"sell_price_lagged_{lag}")
                for lag in PRICE_LAGS
            ]
            + [
                pl.col("sell_price")
                .shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
            + [
                pl.col("sell_price")
                .shift(1)
                .rolling_std(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_std_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
        )
        .drop("sell_price")
    )

    # Join id, lagged, rolling, price, and calendar dfs
    test_df = (
        test_df.drop("sales")
        .join(
            other=sales_features_test_df.drop("sales"),
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        .join(
            other=(
                calendar_features_df.drop("d", "date")
                .join(
                    other=event_features_df.drop("d", "date"),
                    on=["wm_yr_wk", "d_index"],
                    how="inner",
                    validate="1:1",
                )
                .join(
                    other=election_features_df.drop("d", "date"),
                    on=["wm_yr_wk", "d_index"],
                    how="inner",
                    validate="1:1",
                )
            ),
            on="d_index",
            how="inner",
            validate="m:1"
        )
        .join(
            other=price_features_test_df,
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        # Keep inner join semantics: drop series without a price at this timestamp
        .filter(pl.col("sell_price").is_not_null())
    )

    # Make predictions
    # TODO: Take predictions from model and actually join with 
    # FORECAST_IDS from test_df
    y_hat = np.random.randint(0, 10, size=(len(FORECAST_IDS), ))
    y_hat_df = (
        pl.DataFrame({"id": FORECAST_IDS, "sales": y_hat})
        .with_columns(
            sales=pl.col("sales").cast(pl.Int16),
            d_index=pl.lit(d_index).cast(pl.Int16),
            F=pl.format("F{}", f_index),
        )
    )

    # Save predictions
    all_forecast_dfs.append(y_hat_df)


In [21]:
sample_submission_df.head()

id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""HOBBIES_1_001_CA_1_validation""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""HOBBIES_1_002_CA_1_validation""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""HOBBIES_1_003_CA_1_validation""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""HOBBIES_1_004_CA_1_validation""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""HOBBIES_1_005_CA_1_validation""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [22]:
final_forecast_df = (
    pl.concat(all_forecast_dfs)
    .with_columns(sales=pl.col("sales").cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

submission_df = (
    sample_submission_df
    .join(
        other=final_forecast_df,
        on="id",
        how="left",
        suffix="_model",
        validate="1:1"
    )
    .with_columns(
        pl.coalesce(pl.col(f"{c}_model"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(sample_submission_df.columns)
)

In [23]:
submission_df.tail()

id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""FOODS_3_823_WI_3_evaluation""",5,6,0,4,2,1,2,3,5,2,5,9,2,9,8,1,0,5,9,2,3,8,9,6,7,3,9,7
"""FOODS_3_824_WI_3_evaluation""",6,8,1,7,2,3,2,0,4,8,6,9,7,6,1,3,9,6,7,1,5,3,3,6,7,1,1,0
"""FOODS_3_825_WI_3_evaluation""",8,7,2,0,4,9,1,6,6,0,6,7,1,9,7,8,7,4,8,6,3,2,8,5,5,6,4,5
"""FOODS_3_826_WI_3_evaluation""",8,4,7,4,6,8,2,5,7,3,4,0,7,5,1,3,1,5,8,9,3,6,1,9,9,1,5,3
"""FOODS_3_827_WI_3_evaluation""",6,8,6,0,0,4,1,3,3,5,6,1,3,4,0,6,3,1,0,3,7,2,1,0,5,7,8,2
